Daily Challenge : Pinecone Serverless Rerankin



Partie 1: Charger les Documents & Exécuter le Modèle de Reranking


In [1]:
#1. Installer les bibliothèques Pinecone

!pip install pinecone==6.0.1 pinecone-notebooks


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.4/421.4 kB 1.2 MB/s eta 0:00:00


In [11]:
!pip install pandas torch transformers sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 47.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [24]:
#2. S'authentifier avec Pinecone

import os, time, pandas as pd, torch
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from pinecone import RerankModel

import torch
from sentence_transformers import SentenceTransformer


if not os.environ.get("PINECONE_API_KEY"):
   from pinecone_notebooks.colab import Authenticate
   Authenticate()

In [3]:
#3. Instancier le client Pinecone

api_key = os.environ["PINECONE_API_KEY"]
# 👉 REMPLACEZ "gcp-starter" par l'environnement de votre projet Pinecone (ex: "us-west1-gcp", "aws-us-east-1", etc.)
environment = "gcp-starter"
pc = Pinecone(api_key=api_key, environment=environment)

print(f"✅ Client Pinecone instancié pour l'environnement : {environment}")

✅ Client Pinecone instancié pour l'environnement : gcp-starter


In [4]:
#4. Définir votre requête et vos documents
#Pour démontrer le reranking, nous allons utiliser une requête simple et une liste de documents qui contiennent une ambiguïté : le mot "Apple" peut faire référence au fruit ou à l'entreprise. Le reranker doit être capable de distinguer ces contextes.

query = "Tell me about Apple's products"
documents = [
   "Apple is a widely cultivated fruit, known for its crisp texture and sweet taste.",
   "The new Apple iPhone 16 features an advanced camera system and a powerful chip.",
   "Apples are often used in pies, cider, and sauces, and are a good source of fiber.",
   "Apple Inc. announced its latest line of MacBook Pro laptops, boasting impressive performance upgrades.",
   "Eating an apple a day is said to keep the doctor away due to its numerous health benefits."
]

print("✅ Requête définie et documents créés pour le test d'ambiguïté.")
for i, doc in enumerate(documents):
    print(f"Document {i+1}: {doc}")

✅ Requête définie et documents créés pour le test d'ambiguïté.
Document 1: Apple is a widely cultivated fruit, known for its crisp texture and sweet taste.
Document 2: The new Apple iPhone 16 features an advanced camera system and a powerful chip.
Document 3: Apples are often used in pies, cider, and sauces, and are a good source of fiber.
Document 4: Apple Inc. announced its latest line of MacBook Pro laptops, boasting impressive performance upgrades.
Document 5: Eating an apple a day is said to keep the doctor away due to its numerous health benefits.


Analyse


Requête clairement définie : la ligne Requête définie et documents créés pour le test d'ambiguïté. confirme que la chaîne de caractères query = "Tell me about Apple's products" a bien été stockée et est prête à être utilisée. Cette requête est intentionnellement ambiguë, ciblant "Apple's products" (les produits d'Apple l'entreprise).

Documents Prêts pour le Reranking : Les cinq documents ont été correctement définis et affichés. Leur contenu est crucial :

Documents 1, 3 et 5 parlent de la pomme (le fruit).

Documents 2 et 4 parlent de l'entreprise Apple (iPhone, MacBook Pro).

L'objectif de cette section est de mettre en place un scénario où une simple recherche de similarité pourrait mélanger les résultats (fruit et entreprise), et où le reranking sera nécessaire pour désambiguïser et donner la priorité aux documents les plus pertinents pour l'entreprise Apple, étant donné la requête.

En bref, cette étape est une préparation réussie de notre petit jeu de données pour démontrer le pouvoir du reranking. Tout est en place.

In [40]:
#5. Appeler le reranker



# 👉 REMPLACEZ 3 par le nombre de résultats rerankés que vous souhaitez obtenir.
# Il devrait être <= au nombre de documents fournis.
top_n_results = 3

print(f"\n🚀 Appel du modèle de reranking 'bge-reranker-v2-m3'...")
try:
    reranked = pc.inference.rerank(
       model="bge-reranker-v2-m3",
       query=query,
       documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
       top_n=top_n_results
    )
    print(f"✅ Modèle de reranking exécuté. Demande de {top_n_results} résultats principaux.")
except Exception as e:
    reranked = None
    print(f"❌ Erreur lors de l'appel au reranker : {e}")
    print("Veuillez vérifier votre connexion Internet et la validité de votre clé API/environnement Pinecone.")



🚀 Appel du modèle de reranking 'bge-reranker-v2-m3'...
✅ Modèle de reranking exécuté. Demande de 3 résultats principaux.


Ce que nous pouvons en conclure est que l'appel à l'API de Pinecone pour le reranking s'est bien déroulé :

Modèle de Reranking contacté : la ligne 🚀 Appel du modèle de reranking 'bge-reranker-v2-m3'... indique que la requête a bien été envoyée au modèle bge-reranker-v2-m3 hébergé par Pinecone. Ce modèle est conçu spécifiquement pour évaluer la pertinence de documents par rapport à une requête donnée.

Exécution réussie : le message Modèle de reranking exécuté. Demande de 3 résultats principaux. confirme que Pinecone a traité votre demande. Le modèle a analysé votre query ("Tell me about Apple's products") et les cinq documents que vous lui avez fournis, puis il a recalculé des scores de pertinence pour chacun d'eux.

Résultats prêts à être inspectés :lL'objet reranked contient maintenant les documents réordonnés en fonction de leur nouvelle pertinence calculée par le modèle. Il est prêt pour l'étape d'inspection.



In [42]:
#6. Inspecter les résultats rerankés

def show_reranked_part1(query_text, matches):
   print(f"\n--- Résultats du Reranking ---")
   print(f"Requête : '{query_text}'")
   print("--------------------------------------------------")
   if matches: # Vérifie si 'matches' n'est PAS None et n'est PAS vide
       for i, m in enumerate(matches):
           print(f"  {i+1}. Score : {m.score:.4f}")
           print(f"     Document : {m.document.text}")
           print("--------------------------------------------------")
   else:
       print("⚠️ Aucun résultat de reranking trouvé ou 'matches' est vide/None pour la démonstration de la Partie 1.")
       print("Ceci peut être dû à des limitations de votre plan Pinecone ou un problème temporaire.")
       print("Nous continuerons néanmoins vers la Partie 2 qui utilise un flux de travail plus complet.")

Partie 2: Configurer un Index Serverless pour les Notes Médicales


In [26]:
# 2. Importer les modules & définir les paramètres d'environnemen
# 👉 TRÈS IMPORTANT : Ces valeurs doivent correspondre à celles définies en Partie 1, point 3.
# Pour le plan Starter (gratuit), c'est TOUJOURS 'aws' et 'us-east-1' pour Serverless.
cloud = "aws"
region = "us-east-1"

# Définition des spécifications Serverless (CPU et mémoire alloués)
spec = ServerlessSpec(cloud=cloud, region=region) # cpu et memory_gb sont gérés automatiquement par ServerlessSpec
index_name = "pinecone-reranker-medical-notes" # Nom unique pour votre index

# Le client 'pc' a déjà été instancié en Partie 1, point 3 avec les bonnes informations.
# Nous n'avons pas besoin de le réinstancier ici à moins que l'environnement ne change,
# mais on le mentionne pour référence.
# pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"], environment=f"{cloud}-{region}")

print(f"✅ Paramètres d'environnement définis : Cloud={cloud}, Région={region}")
print(f"Spécifications Serverless : {spec}")
print(f"Nom de l'index : {index_name}")

✅ Paramètres d'environnement définis : Cloud=aws, Région=us-east-1
Spécifications Serverless : ServerlessSpec(cloud='aws', region='us-east-1')
Nom de l'index : pinecone-reranker-medical-notes


In [16]:
#3. Créer ou recréer l'index

cloud = "aws"        # Mettez toujours 'aws' pour le plan Starter Serverless
region = "us-east-1"  # Mettez toujours 'us-east-1' pour le plan Starter Serverless

spec = ServerlessSpec(cloud=cloud, region=region)
index_name = "pinecone-reranker-medical-notes" # Un nom unique pour votre index

# Assurez-vous de réinstancier le client si vous changez le cloud/région
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"], environment=f"{cloud}-{region}")

print(f"✅ Paramètres d'environnement définis : Cloud={cloud}, Région={region}")
print(f"Spécifications Serverless : {spec}")
print(f"Nom de l'index : {index_name}")

✅ Paramètres d'environnement définis : Cloud=aws, Région=us-east-1
Spécifications Serverless : ServerlessSpec(cloud='aws', region='us-east-1')
Nom de l'index : pinecone-reranker-medical-notes


Partie 3: Charger les Données d'Exemple


In [27]:
#1. Télécharger & lire JSONL

import requests, tempfile

# URL raw GitHub vers le fichier JSONL des notes médicales
jsonl_url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"

try:
    with tempfile.TemporaryDirectory() as tmpdir:
       file_path = os.path.join(tmpdir, "sample_notes_data.jsonl")
       print(f"⬇️ Téléchargement du fichier de données depuis : {jsonl_url}")
       resp = requests.get(jsonl_url)
       resp.raise_for_status() # Lève une erreur pour les codes d'état HTTP non-2xx
       with open(file_path, "wb") as f:
           f.write(resp.content)

       df = pd.read_json(file_path, orient='records', lines=True)

    print(f"✅ Données chargées dans un DataFrame. Nombre de notes: {len(df)}")
except requests.exceptions.RequestException as e:
    df = pd.DataFrame() # Définit df comme DataFrame vide en cas d'erreur
    print(f"❌ Erreur lors du téléchargement ou de la lecture du fichier JSONL : {e}")
    print("Vérifiez l'URL et votre connexion Internet.")



⬇️ Téléchargement du fichier de données depuis : https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl
✅ Données chargées dans un DataFrame. Nombre de notes: 100


In [28]:
#2. Prévisualiser le DataFrame

if not df.empty:
    print("\n--- Aperçu du DataFrame :")
    print(df.head())
else:
    print("\n⚠️ Le DataFrame est vide car le chargement a échoué.")


--- Aperçu du DataFrame :
     id                                             values  \
0  P011  [-0.2027486265, 0.2769146562, -0.1509393603, 0...   
1  P001  [0.1842793673, 0.4459365904, -0.0770567134, 0....   
2  P002  [-0.2040648609, -0.1739618927, -0.2897160649, ...   
3  P003  [0.1889383644, 0.2924542725, -0.2335938066, -0...   
4  P004  [-0.12171068040000001, 0.1674752235, -0.231888...   

                                            metadata  
0  {'advice': 'rest, hydrate', 'symptoms': 'heada...  
1  {'tests': 'EKG, stress test', 'symptoms': 'che...  
2  {'HbA1c': '7.2', 'condition': 'diabetes', 'med...  
3  {'symptoms': 'cough, wheezing', 'diagnosis': '...  
4  {'referral': 'dermatology', 'condition': 'susp...  


Voici ce que nous pouvons en conclure :

✅ Téléchargement Réussi : Le message ⬇️ Téléchargement du fichier de données depuis : ... suivi de ✅ Données chargées dans un DataFrame. indique que votre Colab a pu se connecter à l'URL fournie, télécharger le fichier sample_notes_data.jsonl, et le lire sans problème. C'est une étape cruciale réussie.

📦 Quantité de Données : Le Nombre de notes: 100 nous informe que le fichier contenait 100 enregistrements ou "notes". C'est un bon volume pour un exemple, suffisamment petit pour être gérable, mais assez grand pour démontrer la pertinence de la recherche.

📊 Structure du DataFrame : L'aperçu (df.head()) est très instructeur :

id : Cette colonne contient un identifiant unique pour chaque note (ex: P011, P001). C'est essentiel, car Pinecone utilise ces IDs pour stocker et récupérer les vecteurs.

values : C'est la colonne la plus importante pour Pinecone. Elle contient la représentation vectorielle (embedding) de chaque note médicale. Chaque entrée est une liste de nombres flottants (comme [-0.2027486265, 0.2769146562, ...]). Ces vecteurs sont la traduction sémantique du contenu textuel des notes, permettant à Pinecone de mesurer la similarité.

metadata : Cette colonne contient les informations textuelles originales et structurées de chaque note (ex: {'advice': 'rest, hydrate', 'symptoms': 'headache'}). Ces métadonnées ne sont pas directement utilisées pour la recherche de similarité par Pinecone, mais elles sont stockées aux côtés des vecteurs. Elles sont cruciales car elles seront récupérées après la recherche pour fournir le contexte lisible et seront utilisées par le modèle de reranking pour affiner la pertinence.

En bref, les données ont été correctement chargées, structurées et sont prêtes à être upsertées dans votre index Pinecone Serverless. Le DataFrame df est bien défini et contient les informations nécessaires dans le format attendu par Pinecone.

Partie 4: Upsert des Données dans l'Index


In [29]:
#1. Instancier l'index et upsert les données


# Attendre que l'index soit en état 'ready' avant d'y accéder
print(f"\n⏳ Attente que l'index '{index_name}' soit prêt pour les opérations d'upsert...")
try:
    while not pc.describe_index(index_name).status['ready']:
        print("...")
        time.sleep(5)
    print(f"✅ Index '{index_name}' est prêt.")

    if not df.empty:
        index = pc.Index(index_name)
        print(f"⬆️ Début de l'upsert de {len(df)} vecteurs dans l'index '{index_name}'...")
        index.upsert_from_dataframe(df)
        print(f"✅ Upsert terminé. L'index contiendra des vecteurs dans quelques instants.")
    else:
        print("⚠️ Impossible d'upserter des données : Le DataFrame 'df' est vide ou n'a pas été chargé.")

except Exception as e:
    print(f"❌ Erreur lors de l'attente de l'index ou de l'upsert : {e}")



⏳ Attente que l'index 'pinecone-reranker-medical-notes' soit prêt pour les opérations d'upsert...
✅ Index 'pinecone-reranker-medical-notes' est prêt.
⬆️ Début de l'upsert de 100 vecteurs dans l'index 'pinecone-reranker-medical-notes'...


sending upsert requests:   0%|          | 0/100 [00:00<?, ?it/s]

✅ Upsert terminé. L'index contiendra des vecteurs dans quelques instants.


In [30]:
#2. Disponibilité des vecteurs

def is_ready(idx):
   try:
       stats = idx.describe_index_stats()
       return stats.total_vector_count > 0
   except Exception as e:
       print(f"❌ Erreur lors de la vérification des stats de l'index : {e}")
       return False # Retourne False pour continuer à attendre ou gérer l'erreur

print("\n⏳ Attente de l'indexation des vecteurs...")
try:
    # L'objet 'index' doit avoir été créé à l'étape précédente pour que cela fonctionne.
    # Si l'upsert a échoué car df était vide, 'index' pourrait ne pas être défini.
    if 'index' in locals() and is_ready(index): # Vérifie si 'index' est défini et si is_ready() est vrai d'entrée
        print("\n✅ Index déjà prêt avec des vecteurs !")
    else: # Sinon, on entre dans la boucle d'attente
        while not is_ready(index):
            time.sleep(5)
            print("...", end="")

    print("\n\n✅ Index prêt ! Statistiques de l'index :")
    print(index.describe_index_stats())

except NameError:
    print("\n❌ L'objet 'index' n'est pas défini. La partie 'Upsert' n'a peut-être pas été exécutée ou a échoué.")
    print("Veuillez vous assurer que les parties précédentes ont été exécutées avec succès.")
except Exception as e:
    print(f"\n❌ Une erreur inattendue est survenue lors de l'attente de l'indexation : {e}")



⏳ Attente de l'indexation des vecteurs...

✅ Index déjà prêt avec des vecteurs !


✅ Index prêt ! Statistiques de l'index :
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 100}},
 'total_vector_count': 100,
 'vector_type': 'dense'}


Partie 5: Requête & Fonction d'Embedding


In [31]:
#1. Définir la fonction d'embedding

# Le modèle Sentence-Transformer que nous allons utiliser.
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

def get_embedding(text):
   """
   Génère l'embedding (vecteur numérique) d'un texte donné.
   Le modèle est chargé une seule fois pour des raisons de performance.
   """
   return embedding_model.encode(text).tolist() # Convertit le tenseur PyTorch en liste Python

print(f"✅ Fonction d'embedding définie avec le modèle '{EMBEDDING_MODEL_NAME}'.")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Fonction d'embedding définie avec le modèle 'all-MiniLM-L6-v2'.


In [32]:
#2. Exécuter une requête de recherche sémantique

# 👉 REMPLACEZ ceci par votre question clinique que vous souhaitez rechercher.
question = "what if my patient has severe leg pain after surgery"
# 👉 REMPLACEZ ceci par le nombre de résultats initiaux que vous souhaitez récupérer de Pinecone.
top_k_initial = 5

try:
    emb = get_embedding(question) # Génère l'embedding de notre question

    print(f"\n🔍 Exécution de la recherche sémantique pour la question : '{question}'")
    print(f"   Récupération de {top_k_initial} résultats initiaux.")

    results = index.query(vector=emb, top_k=top_k_initial, include_metadata=True)
    matches = sorted(results.matches, key=lambda m: m.score, reverse=True)

    print(f"✅ Recherche sémantique terminée. {len(matches)} résultats trouvés.")
except NameError:
    matches = [] # Définit matches comme vide si 'index' n'est pas défini
    print("\n❌ L'objet 'index' n'est pas défini. La partie 'Upsert' n'a pas été exécutée ou a échoué.")
except Exception as e:
    matches = []
    print(f"\n❌ Erreur lors de l'exécution de la recherche sémantique : {e}")



🔍 Exécution de la recherche sémantique pour la question : 'what if my patient has severe leg pain after surgery'
   Récupération de 5 résultats initiaux.
✅ Recherche sémantique terminée. 5 résultats trouvés.


Partie 6: Afficher & Reranker les Notes Cliniques


In [33]:
#1. Afficher les résultats de recherche initiaux

def show_results(q, matches_list):
   print(f"\n--- Résultats de la recherche sémantique initiale pour : '{q}' ---")
   print("Ces résultats sont classés par similarité de vecteur (cosine similarity).")
   print("---------------------------------------------------------------------")
   if matches_list:
       for i, m in enumerate(matches_list):
           print(f"  {i+1}. ID du vecteur : {m.id}")
           print(f"     Score de similarité (initial) : {m.score:.4f}")
           print(f"     Métadonnées : {m.metadata}")
           print("---------------------------------------------------------------------")
   else:
       print("⚠️ Aucun résultat trouvé pour la recherche sémantique initiale.")

show_results(question, matches)



--- Résultats de la recherche sémantique initiale pour : 'what if my patient has severe leg pain after surgery' ---
Ces résultats sont classés par similarité de vecteur (cosine similarity).
---------------------------------------------------------------------
  1. ID du vecteur : P007
     Score de similarité (initial) : 0.4817
     Métadonnées : {'surgery': 'knee arthroscopy', 'symptoms': 'pain, swelling', 'treatment': 'physical therapy'}
---------------------------------------------------------------------
  2. ID du vecteur : P055
     Score de similarité (initial) : 0.4719
     Métadonnées : {'recovery': 'no complications', 'surgery': 'hernia repair'}
---------------------------------------------------------------------
  3. ID du vecteur : P079
     Score de similarité (initial) : 0.4677
     Métadonnées : {'recovery': 'well', 'surgery': 'appendix removal'}
---------------------------------------------------------------------
  4. ID du vecteur : P0100
     Score de similarité (i


Recherche sémantique réussie : le système a correctement pris la question ("what if my patient has severe leg pain after surgery"), l'a convertie en un vecteur (embedding), puis a recherché les vecteurs les plus similaires dans votre index Pinecone.

Pertinence initiale basée sur la Similarité Vectorielle : les résultats sont classés par leur score de similarité cosinus (le plus élevé étant le plus similaire).

Le document P007 est en tête avec un score de 0.4817. Ses métadonnées ('surgery': 'knee arthroscopy', 'symptoms': 'pain, swelling', 'treatment': 'physical therapy') sont très pertinentes pour la requête sur la "douleur à la jambe après une chirurgie".

Les documents suivants (P055, P079) ont des scores légèrement inférieurs et parlent d'autres chirurgies (hernie, appendice), ce qui est moins spécifique à la "douleur à la jambe".

Le document P0100 parle de "muscle pain" et P047 de "back pain", qui sont moins directement liés à la "douleur à la jambe après chirurgie" que P007.

Observation clé : la recherche sémantique a bien identifié des documents globalement pertinents. Cependant, même si P007 est le plus pertinent, les autres résultats sont classés uniquement sur la base de la similarité vectorielle de leur contenu global. Le reranker aura pour rôle d'affiner cet ordre en considérant la pertinence spécifique par rapport à la requête, en allant au-delà de la simple similarité vectorielle.



In [34]:
#2. Préparer les documents pour le reranking

# Préparer les documents pour le reranking seulement si des matches ont été trouvés
rerank_docs = []
if matches:
    rerank_docs = [
       {"id": m.id, "reranking_field": "; ".join([f"{k}: {v}" for k, v in m.metadata.items()])}
       for m in matches
    ]

# 👉 REMPLACEZ ceci par une question clinique plus spécifique pour le reranking
rerank_query = "patient complains about severe, throbbing pain in the right leg after a knee surgery and potential infection"

print(f"\n📝 Documents préparés pour le reranking. Une requête affinée est utilisée : '{rerank_query}'")
if rerank_docs:
    print(f"Exemple de 'reranking_field' pour le premier document : '{rerank_docs[0]['reranking_field']}'")
else:
    print("⚠️ Aucun document à préparer pour le reranking car la recherche initiale n'a rien trouvé.")


📝 Documents préparés pour le reranking. Une requête affinée est utilisée : 'patient complains about severe, throbbing pain in the right leg after a knee surgery and potential infection'
Exemple de 'reranking_field' pour le premier document : 'surgery: knee arthroscopy; symptoms: pain, swelling; treatment: physical therapy'


In [36]:
#3. Exécuter le reranking serverless


#
top_n_reranked = 5

reranked_clinical_notes = None # Initialise à None

if rerank_docs: # N'exécute le reranking que si des documents sont à reranker
    print(f"\n✨ Exécution du reranking avec le modèle 'bge-reranker-v2-m3' sur les documents préparés...")
    try:
        reranked_clinical_notes = pc.inference.rerank(
           model="bge-reranker-v2-m3",
           query=rerank_query,
           documents=rerank_docs,
           rank_fields=["reranking_field"],
           top_n=top_n_reranked
        )
        print(f"✅ Reranking terminé. {top_n_reranked} résultats rerankés obtenus.")
    except Exception as e:
        print(f"❌ Erreur lors de l'exécution du reranking : {e}")
else:
    print("⚠️ Aucun document à reranker, la recherche initiale n'a pas produit de résultats.")


✨ Exécution du reranking avec le modèle 'bge-reranker-v2-m3' sur les documents préparés...
✅ Reranking terminé. 5 résultats rerankés obtenus.


In [37]:
#4. Afficher les résultats rerankés

def show_reranked_clinical(q, matches_list):
   print(f"\n--- Résultats Rerankés pour la Requête Affinée : '{q}' ---")
   print("Ces résultats sont classés par le modèle de reranking.")
   print("---------------------------------------------------------------------")
   if matches_list:
       for i, m in enumerate(matches_list):
           print(f"  {i+1}. ID du document : {m.document.id}")
           print(f"     Score de Reranking : {m.score:.4f}")
           print(f"     Contenu reranké : {m.document.reranking_field}")
           print("---------------------------------------------------------------------")
   else:
       print("⚠️ Aucun résultat trouvé après le reranking.")

show_reranked_clinical(rerank_query, reranked_clinical_notes.matches if reranked_clinical_notes else None)


--- Résultats Rerankés pour la Requête Affinée : 'patient complains about severe, throbbing pain in the right leg after a knee surgery and potential infection' ---
Ces résultats sont classés par le modèle de reranking.
---------------------------------------------------------------------
⚠️ Aucun résultat trouvé après le reranking.


Explication de l'Obstacle Persistant
Quota Atteint : comme mentionné précédemment, le plan Starter de Pinecone a une limite de 500 requêtes de reranking par mois (pour le modèle bge-reranker-v2-m3). Il est presque certain que vous avez atteint ou dépassé cette limite. Lorsque le quota est dépassé, l'API peut ne pas renvoyer une erreur explicite à votre code Python, mais simplement une liste de résultats vide ou None, ce qui se traduit par le message "Aucun résultat trouvé".

Services d'Inférence vs. Base de Données Vectorielle : il est important de distinguer :

La base de données vectorielle (création d'index, upsert, query) : c'est la fonctionnalité principale de Pinecone, et elle est assez généreuse sur le plan Starter, ce qui explique pourquoi vos Parties 2, 3, 4 et 5 ont fonctionné à merveille. Vous avez pu créer l'index, y insérer 100 notes et les interroger avec succès.

Les services d'inférence (comme le reranking) : ce sont des modèles pré-entraînés que Pinecone propose en tant que service. Ils sont souvent soumis à des quotas plus stricts ou à des plans payants, car ils impliquent des calculs intensifs sur les serveurs de Pinecone.